# Debug a LangGraph Agent with AgentDbg

This tutorial shows how to trace and debug a multi-step LangGraph agent using AgentDbg — no API keys required.

## Why agent debugging is hard

AI agents run multi-step workflows with LLM calls and tool use. When something goes wrong, it's unclear which step failed or why. AgentDbg captures a structured timeline of every run so you can see exactly what happened.

## Setup

Install AgentDbg with LangChain support and LangGraph (run once):

In [1]:
%pip install --upgrade pip -q
%pip install "agentdbg[langchain]" langgraph -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import operator
from typing import Annotated, Any

from agentdbg import trace, AgentDbgLoopAbort, record_llm_call, record_tool_call
from agentdbg.integrations import AgentDbgLangChainCallbackHandler

from langchain_core.language_models.fake import FakeListLLM
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END

from typing_extensions import TypedDict

## Define tools

We'll use three tools: **search** (returns deterministic data), **calculator** (evaluates an expression), and **save_result** (confirms saving). All are deterministic so the tutorial runs without API keys.

In [7]:
@tool
def search(query: str) -> str:
    """Search for information (deterministic stub)."""
    return "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"


@tool
def calculator(expression: str) -> str:
    """Evaluate a numeric expression."""
    return str(eval(expression))


@tool
def save_result(data: str) -> str:
    """Save a result and return confirmation."""
    return f"Saved: {data}"

## Build the LangGraph agent

The graph has an **agent** node (LLM decides next action) and three **tool** nodes. The LLM output is parsed for keywords (`search`, `calculate`, `save`, `DONE`) to route to the next node.

In [8]:
class AgentState(TypedDict, total=False):
    task: str
    llm_output: str
    tool_results: Annotated[list[str], operator.add]
    _llm: Any  # LLM instance; passed in initial state so it flows through


def agent_node(state: AgentState, config: RunnableConfig | None = None):
    config = config or {}
    llm = state.get("_llm")
    task = state.get("task", "")
    results = state.get("tool_results") or []
    context = " ".join(results[-2:]) if results else task
    out = llm.invoke(context, config=config)
    return {"llm_output": out.content if hasattr(out, "content") else str(out)}


def search_node(state: AgentState, config: RunnableConfig | None = None):
    config = config or {}
    result = search.invoke({"query": "quarterly sales"}, config=config)
    return {"tool_results": [result]}


def calc_node(state: AgentState, config: RunnableConfig | None = None):
    config = config or {}
    result = calculator.invoke({"expression": "1.2+1.5+2.0+1.8"}, config=config)
    return {"tool_results": [result]}


def save_node(state: AgentState, config: RunnableConfig | None = None):
    config = config or {}
    data = state.get("tool_results") or []
    to_save = data[-1] if data else ""
    result = save_result.invoke({"data": to_save}, config=config)
    return {"tool_results": [result]}


def route(state: AgentState) -> str:
    out = (state.get("llm_output") or "").lower()
    if "done" in out:
        return END
    if "search" in out:
        return "search_node"
    if "calculate" in out or "calc" in out:
        return "calc_node"
    if "save" in out:
        return "save_node"
    return END

In [9]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("search_node", search_node)
builder.add_node("calc_node", calc_node)
builder.add_node("save_node", save_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route)
builder.add_edge("search_node", "agent")
builder.add_edge("calc_node", "agent")
builder.add_edge("save_node", "agent")
graph = builder.compile()

## Run without tracing

First, run the agent with a **happy path**: scripted LLM responses that go search → calculate → save → DONE.

In [10]:
happy_llm = FakeListLLM(
    responses=[
        "I will search for quarterly sales.",
        "I will calculate the total.",
        "I will save the result.",
        "DONE. Summary: 6.5M total.",
    ]
)
initial = {
    "task": "Get quarterly sales and save the total",
    "llm_output": "",
    "tool_results": [],
    "_llm": happy_llm,
}
result = graph.invoke(initial)
print("Final llm_output:", result.get("llm_output"))
print("Tool results:", result.get("tool_results"))

Final llm_output: DONE. Summary: 6.5M total.
Tool results: ['Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M', '6.5', 'Saved: 6.5']


## Instrument with AgentDbg

Add **`@trace`** to the entrypoint and pass **`AgentDbgLangChainCallbackHandler`** in the config. The handler records every LLM and tool call into the active run.

In [11]:
@trace(name="LangGraph agent (happy path)")
def run_traced(llm, initial_state: dict):
    handler = AgentDbgLangChainCallbackHandler()
    config = {"callbacks": [handler]}
    state = {**initial_state, "_llm": llm}
    return graph.invoke(state, config=config)

In [15]:
run_traced(happy_llm, {"task": "Get quarterly sales and save the total", "llm_output": "", "tool_results": []})
print("Run complete. View with: agentdbg view")

Run complete. View with: agentdbg view


In [16]:
!agentdbg list

run_id  	run_name                    	started_at              	duration_ms	llm_calls	tool_calls	status
ad365863	LangGraph agent (happy path)	2026-03-14T18:30:46.361Z	85         	4        	3         	ok    


## View the timeline

In a **terminal**, run:

```bash
agentdbg view
```

A browser opens at `http://127.0.0.1:8712`. You'll see:

- **Sidebar**: Recent runs (newest first) with name, time, status, duration.
- **Run summary**: Status (OK/ERROR/RUNNING), counts (LLM calls, tools, errors, loop warnings), and filter chips (All, LLM, Tools, Errors, State, Loops).
- **Timeline**: Events in order; expand any event to see payload and meta as JSON.

### What happened in this run

1. **RUN_START** — Trace began.
2. **LLM_CALL** — Agent decided to search.
3. **TOOL_CALL** (search) — Search returned quarterly figures.
4. **LLM_CALL** — Agent decided to calculate.
5. **TOOL_CALL** (calculator) — Total 6.5.
6. **LLM_CALL** — Agent decided to save.
7. **TOOL_CALL** (save_result) — Saved.
8. **LLM_CALL** — Agent said DONE.
9. **RUN_END** — Trace finished successfully.

## The looping agent

Next we simulate an agent **stuck in a loop**: the LLM keeps saying "search again" so the graph keeps calling the search tool and re-asking the LLM. We set AgentDbg's loop-detection env vars so a repeated pattern is detected quickly.

In [17]:
os.environ.setdefault("AGENTDBG_LOOP_WINDOW", "6")
os.environ.setdefault("AGENTDBG_LOOP_REPETITIONS", "3")

# LLM that keeps saying "search" for several turns, then DONE so the run eventually finishes
looping_llm = FakeListLLM(
    responses=["search again"] * 5 + ["DONE."]
)

In [18]:
@trace(name="LangGraph agent (looping, no guardrail)")
def run_looping_no_guardrail():
    handler = AgentDbgLangChainCallbackHandler()
    config = {"callbacks": [handler]}
    state = {"task": "Find sales", "llm_output": "", "tool_results": [], "_llm": looping_llm}
    return graph.invoke(state, config=config)


run_looping_no_guardrail()
print("Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.")

Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.


In [19]:
!agentdbg list

run_id  	run_name                               	started_at              	duration_ms	llm_calls	tool_calls	status
0ec72238	LangGraph agent (looping, no guardrail)	2026-03-14T18:32:49.848Z	129        	6        	5         	ok    
ad365863	LangGraph agent (happy path)           	2026-03-14T18:30:46.361Z	85         	4        	3         	ok    


## Understanding LOOP_WARNING

AgentDbg's loop detector looks at the **last N events** (the "window"). If the same **signature pattern** (e.g. TOOL_CALL:search + LLM_CALL:fake) repeats **3 times** in a row, it emits a **LOOP_WARNING** event. In the viewer, use the **Loops** filter to see it; the event payload shows the pattern and which event IDs formed the evidence.

Here the agent called **search** and the **LLM** three times in a row with no progress. AgentDbg flags that so you can fix the prompt or logic instead of burning more tokens.

## Fix with stop_on_loop

To **prevent** a runaway loop, use the **stop_on_loop** guardrail. When a LOOP_WARNING is emitted and the repetition count meets the minimum, AgentDbg aborts the run and raises **AgentDbgLoopAbort**. The trace is still saved so you can inspect what happened.

Note: When using the LangChain callback handler, the framework may catch exceptions from callbacks, so the graph can keep running until its recursion limit. To see **stop_on_loop** actually abort a run, we use a small loop with **record_tool_call** / **record_llm_call** (same idea as the guardrails demo).

In [20]:
# Pure-Python loop so the guardrail exception propagates (callback handlers may swallow it)
@trace(
    name="stop_on_loop demo",
    stop_on_loop=True,
    stop_on_loop_min_repetitions=3,
)
def run_loop_that_aborts():
def run_looping_no_guardrail():
    handler = AgentDbgLangChainCallbackHandler()
    config = {"callbacks": [handler]}
    state = {"task": "Find sales", "llm_output": "", "tool_results": [], "_llm": looping_llm}
    return graph.invoke(state, config=config)



try:
    run_loop_that_aborts()
    print("Unexpected: run did not abort")
except AgentDbgLoopAbort as e:
    print(f"AgentDbg stopped the agent: {e}")
    print("The full trace is saved; open it with: agentdbg view")

AgentDbg stopped the agent: guardrail stop_on_loop: repetitions 3 >= stop_on_loop_min_repetitions 3
The full trace is saved; open it with: agentdbg view


The run was **aborted** after the repeated pattern was detected. The timeline still contains all events up to and including the LOOP_WARNING, so you can see exactly which tool and LLM calls formed the loop and adjust your agent accordingly.

## Next steps

- Run **`agentdbg view`** in your terminal to explore the traces from this notebook.
- See the [README](https://github.com/AgentDbg/AgentDbg/blob/main/README.md) for more guardrails, storage, and integrations.
- Try the examples: `examples/langchain/minimal.py`, `examples/demo/guardrails.py`.